In [1]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess

In [2]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

Groq API Key exists and begins gsk_uqLa


In [3]:
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
models = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct-0905",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b"
]

clients = {model: groq_client for model in models}

In [5]:
from system_info import retrieve_system_info
system_info = retrieve_system_info()
print(system_info)

{'os': {'system': 'Linux', 'arch': 'x86_64', 'release': '6.6.87.2-microsoft-standard-WSL2', 'version': '#1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025', 'kernel': '6.6.87.2-microsoft-standard-WSL2', 'distro': {'name': 'Ubuntu 24.04.3 LTS', 'version': '24.04'}, 'wsl': True, 'rosetta2_translated': False, 'target_triple': 'x86_64-linux-gnu'}, 'package_managers': ['apt'], 'cpu': {'brand': 'Intel(R) Core(TM) i5-6200U CPU @ 2.30GHz', 'cores_logical': 4, 'cores_physical': 2, 'simd': ['AVX', 'AVX2', 'FMA', 'SSE4_2']}, 'toolchain': {'compilers': {'gcc': 'gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0', 'g++': 'g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0', 'clang': '', 'msvc_cl': ''}, 'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 4.3'}, 'linkers': {'ld_lld': ''}}}


In [6]:
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "-flto", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [7]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [10]:
def port(model, python):
    client = clients[model]
    kwargs = {
        "model": model,
        "messages": messages_for(python)
    }
    
    if model == "openai/gpt-oss-120b":
        kwargs["reasoning_effort"] = "medium"
    elif model == "qwen/qwen3-32b":
        kwargs["reasoning_effort"] = "None"
    
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    return reply

In [11]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output

In [12]:
def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [13]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [14]:
with gr.Blocks(title="Port from Python to C++") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label="C++ (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row():
        python_run = gr.Button("Run Python")
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button("Port to C++")
        cpp_run = gr.Button("Run C++")

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8)
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label="C++ result", lines=8)

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


gio: http://127.0.0.1:7861/: Operation not supported


gpt-oss-0.000849s <br>
llama 3.1-8b - error <br>
llama 3.3-70b - error <br>
llama 4- error<br>
kimi-k2 - error<br>
qwen error<br>